<a href="https://colab.research.google.com/github/Dana-El/CCI-HW4/blob/main/BioBERT_Disease_NER_KHCC_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧬 BioBERT Fine-Tuning for Disease NER — KHCC Tumor Registry

**King Hussein Cancer Center | Clinical NLP Pipeline**

This notebook fine-tunes [BioBERT](https://huggingface.co/dmis-lab/biobert-v1.1) for **disease Named Entity Recognition (NER)** on pathology-style tumor registry text.

### Workflow
1. **Baseline** — Run `d4data/biomedical-ner-all` on 5 KHCC pathology samples
2. **Dataset** — Load NCBI Disease corpus via HuggingFace Parquet files
3. **Tokenize** — Sub-word label alignment for BERT
4. **Evaluate (before)** — Entity-level seqeval F1 on random head
5. **Fine-tune** — HuggingFace Trainer on NCBI Disease
6. **Evaluate (after)** — Before/after delta table
7. **Inference** — Side-by-side comparison on same pathology samples
8. **Stretch** — SQLite JSON export with entities, scores & latency
9. **Discussion** — BioBERT vs OpenAI at KHCC

---
> ⚠️ **Before running:** Go to `Runtime → Change runtime type → T4 GPU`

## 0. Setup & Installation

In [1]:
!pip install -q transformers datasets seqeval accelerate evaluate

import warnings, os, json, time, sqlite3
import numpy as np
import pandas as pd
from datetime import datetime

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Device: {device}")
if device == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00
✅ Device: cuda
   GPU: Tesla T4
   VRAM: 15.6 GB


In [2]:
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    pipeline, TrainingArguments, Trainer,
    DataCollatorForTokenClassification,
)
from datasets import load_dataset, Dataset, DatasetDict
import evaluate

print("✅ All imports successful")

✅ All imports successful


## 1. KHCC Pathology Samples

Five representative tumor registry samples covering major cancer types seen at KHCC.

In [3]:
PATHOLOGY_SAMPLES = [
    {
        "id": "KHCC-001", "site": "Breast",
        "text": (
            "Sections show invasive ductal carcinoma, grade III, with extensive lymphovascular invasion. "
            "There is associated ductal carcinoma in situ of solid and cribriform types. "
            "Two of fifteen axillary lymph nodes are positive for metastatic carcinoma. "
            "Background breast shows fibrocystic changes with apocrine metaplasia."
        ),
        "gold_entities": ["invasive ductal carcinoma", "lymphovascular invasion",
                          "ductal carcinoma in situ", "metastatic carcinoma", "fibrocystic changes"]
    },
    {
        "id": "KHCC-002", "site": "Colon",
        "text": (
            "Moderately differentiated adenocarcinoma of the sigmoid colon invading through the muscularis "
            "propria into pericolorectal fat (pT3). Perineural invasion is identified. "
            "Twelve lymph nodes examined, four positive for metastatic adenocarcinoma (pN2). "
            "Proximal and distal resection margins are negative. Microsatellite stable by IHC."
        ),
        "gold_entities": ["adenocarcinoma", "perineural invasion", "metastatic adenocarcinoma"]
    },
    {
        "id": "KHCC-003", "site": "Lung",
        "text": (
            "Right upper lobe wedge resection: Lung adenocarcinoma with lepidic, acinar, and papillary "
            "patterns (lepidic predominant). Visceral pleural invasion present. "
            "No lymphovascular invasion identified. The tumor measures 2.8 cm in greatest dimension. "
            "Background lung shows emphysematous changes and mild chronic inflammation."
        ),
        "gold_entities": ["adenocarcinoma", "visceral pleural invasion",
                          "lymphovascular invasion", "emphysematous changes", "chronic inflammation"]
    },
    {
        "id": "KHCC-004", "site": "Kidney",
        "text": (
            "Nephrectomy specimen: Clear cell renal cell carcinoma, Fuhrman grade 3, "
            "with tumor necrosis (>50%). Adrenal gland involvement by direct extension. "
            "Renal vein thrombosis with tumor thrombus. Gerota's fascia is intact. "
            "No evidence of sarcomatoid or rhabdoid differentiation."
        ),
        "gold_entities": ["clear cell renal cell carcinoma", "tumor necrosis",
                          "adrenal gland involvement", "renal vein thrombosis"]
    },
    {
        "id": "KHCC-005", "site": "Brain",
        "text": (
            "Brain biopsy — right frontal lobe: Glioblastoma multiforme (WHO Grade IV) "
            "with microvascular proliferation and geographic necrosis. "
            "EGFR amplification present; MGMT promoter methylation status: unmethylated. "
            "IDH1/IDH2 wild-type. TERT promoter mutation detected. "
            "Tumor infiltrates adjacent white matter with reactive astrocytosis."
        ),
        "gold_entities": ["glioblastoma multiforme", "microvascular proliferation",
                          "geographic necrosis", "reactive astrocytosis"]
    }
]
print(f"✅ Loaded {len(PATHOLOGY_SAMPLES)} KHCC pathology samples")
for s in PATHOLOGY_SAMPLES:
    print(f"  [{s['id']}] {s['site']}: {len(s['text'])} chars, {len(s['gold_entities'])} gold entities")

✅ Loaded 5 KHCC pathology samples
  [KHCC-001] Breast: 312 chars, 5 gold entities
  [KHCC-002] Colon: 329 chars, 3 gold entities
  [KHCC-003] Lung: 319 chars, 5 gold entities
  [KHCC-004] Kidney: 272 chars, 4 gold entities
  [KHCC-005] Brain: 329 chars, 4 gold entities


## 2. Baseline: `d4data/biomedical-ner-all`

Run the off-the-shelf biomedical NER model on all 5 samples **before any fine-tuning**.

In [4]:
print("⏳ Loading d4data/biomedical-ner-all baseline model...")
baseline_model_name = "d4data/biomedical-ner-all"
baseline_tokenizer = AutoTokenizer.from_pretrained(baseline_model_name)
baseline_model = AutoModelForTokenClassification.from_pretrained(baseline_model_name)

baseline_pipe = pipeline(
    "ner", model=baseline_model, tokenizer=baseline_tokenizer,
    aggregation_strategy="simple",
    device=0 if device == 'cuda' else -1
)
print(f"✅ Baseline model loaded")
print(f"   Labels: {list(baseline_model.config.id2label.values())}")

⏳ Loading d4data/biomedical-ner-all baseline model...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/266M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

✅ Baseline model loaded
   Labels: ['O', 'B-Activity', 'B-Administration', 'B-Age', 'B-Area', 'B-Biological_attribute', 'B-Biological_structure', 'B-Clinical_event', 'B-Color', 'B-Coreference', 'B-Date', 'B-Detailed_description', 'B-Diagnostic_procedure', 'B-Disease_disorder', 'B-Distance', 'B-Dosage', 'B-Duration', 'B-Family_history', 'B-Frequency', 'B-Height', 'B-History', 'B-Lab_value', 'B-Mass', 'B-Medication', 'B-Non[biological](Detailed_description', 'B-Nonbiological_location', 'B-Occupation', 'B-Other_entity', 'B-Other_event', 'B-Outcome', 'B-Personal_[back](Biological_structure', 'B-Personal_background', 'B-Qualitative_concept', 'B-Quantitative_concept', 'B-Severity', 'B-Sex', 'B-Shape', 'B-Sign_symptom', 'B-Subject', 'B-Texture', 'B-Therapeutic_procedure', 'B-Time', 'B-Volume', 'B-Weight', 'I-Activity', 'I-Administration', 'I-Age', 'I-Area', 'I-Biological_attribute', 'I-Biological_structure', 'I-Clinical_event', 'I-Color', 'I-Coreference', 'I-Date', 'I-Detailed_description', '

In [5]:
def run_ner_pipeline(samples, pipe, score_threshold=0.70):
    results = []
    for sample in samples:
        t0 = time.time()
        preds = pipe(sample["text"])
        latency_ms = (time.time() - t0) * 1000
        seen, entities = set(), []
        for e in preds:
            word = e["word"].strip()
            if word.lower() not in seen and e["score"] >= score_threshold:
                seen.add(word.lower())
                entities.append({
                    "word": word, "entity_group": e["entity_group"],
                    "score": round(e["score"], 3),
                    "start": e["start"], "end": e["end"]
                })
        gold = {g.lower() for g in sample["gold_entities"]}
        pred_words = {e["word"].lower() for e in entities}
        matched = sum(1 for g in gold if any(g in p or p in g for p in pred_words))
        results.append({
            "id": sample["id"], "site": sample["site"], "text": sample["text"],
            "entities": entities, "gold_entities": sample["gold_entities"],
            "matched": matched, "total_gold": len(gold),
            "latency_ms": round(latency_ms, 1)
        })
    return results

print("⏳ Running baseline on 5 KHCC pathology samples...")
baseline_results = run_ner_pipeline(PATHOLOGY_SAMPLES, baseline_pipe)
print("✅ Baseline inference complete")

⏳ Running baseline on 5 KHCC pathology samples...
✅ Baseline inference complete


In [6]:
for r in baseline_results:
    print(f"\n{'='*65}")
    print(f"📋 {r['id']} | {r['site']}")
    print(f"TEXT: {r['text'][:110]}...")
    print(f"\n🏷️  DETECTED ({len(r['entities'])} entities):")
    for e in r['entities']:
        print(f"  {e['entity_group']:20s} | {e['word']:35s} | {e['score']:.3f}")
    recall = r['matched'] / r['total_gold']
    print(f"\n🥇 Gold: {r['gold_entities']}")
    print(f"✅ Partial recall: {r['matched']}/{r['total_gold']} ({recall:.0%}) | ⏱ {r['latency_ms']} ms")

avg_recall = np.mean([r['matched']/r['total_gold'] for r in baseline_results])
print(f"\n📊 Average partial recall across 5 samples: {avg_recall:.0%}")


📋 KHCC-001 | Breast
TEXT: Sections show invasive ductal carcinoma, grade III, with extensive lymphovascular invasion. There is associate...

🏷️  DETECTED (15 entities):
  Detailed_description | invasive                            | 1.000
  Disease_disorder     | ductal carcino                      | 0.794
  Biological_structure | l                                   | 0.939
  Disease_disorder     | ductal carcinoma                    | 1.000
  Detailed_description | solid                               | 1.000
  Biological_structure | cr                                  | 0.931
  Detailed_description | ##ib                                | 0.733
  Biological_structure | ax                                  | 1.000
  Biological_structure | ##illa                              | 0.827
  Biological_structure | ##ym                                | 0.996
  Biological_structure | nodes                               | 0.988
  Detailed_description | meta                                | 1.000
  

### 2.1 Baseline Gap Analysis

| Gap Category | Example | Why Missed |
|---|---|---|
| **Multi-word disease terms** | `invasive ductal carcinoma` | Subword boundaries confuse span grouping |
| **Histological modifiers** | `lepidic predominant` | Not in training distribution |
| **Staging terminology** | `perineural invasion` | Treated as anatomy, not disease |
| **Molecular pathology** | `geographic necrosis` | Out of label schema scope |

Fine-tuning on NCBI Disease will specifically improve **disease span detection** with the `B-Disease / I-Disease` label schema.

## 3. Load NCBI Disease Dataset via Parquet

⚠️ **Do NOT use** `load_dataset("ncbi_disease")` or `trust_remote_code=True` — the legacy loading scripts are broken. Load directly from HuggingFace's auto-converted Parquet files.

In [7]:
# ⚠️ CRITICAL: Load from Parquet — NOT load_dataset("ncbi_disease")
HF_BASE = "https://huggingface.co/datasets/ncbi/ncbi_disease/resolve/refs%2Fconvert%2Fparquet/ncbi_disease"

data_files = {
    "train":      f"{HF_BASE}/train/0000.parquet",
    "validation": f"{HF_BASE}/validation/0000.parquet",
    "test":       f"{HF_BASE}/test/0000.parquet",
}

print("⏳ Loading NCBI Disease dataset from Parquet...")
try:
    raw_datasets = load_dataset("parquet", data_files=data_files)
    print("✅ Loaded from HuggingFace Parquet")
except Exception as e:
    print(f"Parquet URL failed ({e})\nTrying ncbi/ncbi_disease...")
    try:
        raw_datasets = load_dataset("ncbi/ncbi_disease")
        print("✅ Loaded via ncbi/ncbi_disease")
    except Exception as e2:
        print(f"Both failed — creating synthetic dataset for demonstration")
        raw_datasets = None

if raw_datasets:
    print(f"\n📦 Dataset splits:")
    for split, ds in raw_datasets.items():
        print(f"  {split:12s}: {len(ds):5d} samples | columns: {ds.column_names}")

⏳ Loading NCBI Disease dataset from Parquet...


0000.parquet:   0%|          | 0.00/425k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/74.7k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/77.0k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

✅ Loaded from HuggingFace Parquet

📦 Dataset splits:
  train       :  5433 samples | columns: ['id', 'tokens', 'ner_tags']
  validation  :   924 samples | columns: ['id', 'tokens', 'ner_tags']
  test        :   941 samples | columns: ['id', 'tokens', 'ner_tags']


In [8]:
# ⚠️ CRITICAL: Define labels manually — Parquet stores ner_tags as plain integers.
# ClassLabel metadata is NOT present in the auto-converted files.
LABEL_NAMES = ["O", "B-Disease", "I-Disease"]
LABEL2ID = {l: i for i, l in enumerate(LABEL_NAMES)}
ID2LABEL  = {i: l for i, l in enumerate(LABEL_NAMES)}
NUM_LABELS = len(LABEL_NAMES)

print("🏷️  NER label schema (BIO):")
for i, label in enumerate(LABEL_NAMES):
    icon = "⚪" if label == "O" else ("🔵" if label.startswith("B") else "🟣")
    print(f"  {icon} {i} → {label}")

# Build synthetic dataset if remote loading failed
if raw_datasets is None:
    SYNTHETIC = [
        {"tokens": ["Patients","with","Huntington","disease","show","neurodegeneration","."],
         "ner_tags": [0,0,1,2,0,1,0]},
        {"tokens": ["BRCA1","mutations","are","linked","to","breast","cancer","."],
         "ner_tags": [0,0,0,0,0,1,2,0]},
        {"tokens": ["Colorectal","carcinoma","is","a","common","malignancy","."],
         "ner_tags": [1,2,0,0,0,1,0]},
        {"tokens": ["Glioblastoma","multiforme","carries","a","poor","prognosis","."],
         "ner_tags": [1,2,0,0,0,0,0]},
        {"tokens": ["Invasive","ductal","carcinoma","is","the","most","common","breast","tumor","."],
         "ner_tags": [1,2,2,0,0,0,0,0,1,0]},
        {"tokens": ["Clear","cell","renal","cell","carcinoma","accounts","for","75%","of","kidney","tumors","."],
         "ner_tags": [1,2,2,2,2,0,0,0,0,0,1,0]},
        {"tokens": ["Hepatocellular","carcinoma","often","arises","in","cirrhotic","liver","."],
         "ner_tags": [1,2,0,0,0,1,0,0]},
        {"tokens": ["Pancreatic","adenocarcinoma","has","a","5-year","survival","below","10%","."],
         "ner_tags": [1,2,0,0,0,0,0,0,0]},
        {"tokens": ["Acute","lymphoblastic","leukemia","is","common","in","children","."],
         "ner_tags": [1,2,2,0,0,0,0,0]},
        {"tokens": ["Fibrocystic","changes","of","the","breast","are","benign","."],
         "ner_tags": [1,2,0,0,0,0,0,0]},
    ]
    rng = np.random.default_rng(42)
    def make_split(n):
        idx = rng.choice(len(SYNTHETIC), n, replace=True)
        return {"tokens": [SYNTHETIC[i]["tokens"] for i in idx],
                "ner_tags": [SYNTHETIC[i]["ner_tags"] for i in idx]}
    raw_datasets = DatasetDict({
        "train":      Dataset.from_dict(make_split(400)),
        "validation": Dataset.from_dict(make_split(100)),
        "test":       Dataset.from_dict(make_split(100)),
    })
    print("\n✅ Synthetic dataset ready (400/100/100 samples)")

ex = raw_datasets["train"][0]
print(f"\n📝 Example: {ex['tokens']}")
print(f"   Tags:   {ex['ner_tags']}")
print(f"   Labels: {[LABEL_NAMES[t] for t in ex['ner_tags']]}")

🏷️  NER label schema (BIO):
  ⚪ 0 → O
  🔵 1 → B-Disease
  🟣 2 → I-Disease

📝 Example: ['Identification', 'of', 'APC2', ',', 'a', 'homologue', 'of', 'the', 'adenomatous', 'polyposis', 'coli', 'tumour', 'suppressor', '.']
   Tags:   [0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 2, 2, 0, 0]
   Labels: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-Disease', 'I-Disease', 'I-Disease', 'I-Disease', 'O', 'O']


## 4. Tokenization with Sub-Word Label Alignment

BERT wordpiece splits clinical terms across multiple sub-tokens. We assign:
- The NER label to the **first sub-token** of each word
- `-100` to all continuation sub-tokens (ignored by the cross-entropy loss)

In [9]:
MODEL_NAME = "dmis-lab/biobert-v1.1"
print(f"⏳ Loading BioBERT tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"✅ Tokenizer loaded (vocab: {tokenizer.vocab_size:,})")

# Demonstrate sub-word splitting on a clinical term
demo = "nephroblastoma"
print(f"\n📝 Sub-word demo: '{demo}' → {tokenizer.tokenize(demo)}")
print("   Only the FIRST sub-token gets the NER label; rest get -100")

⏳ Loading BioBERT tokenizer...


config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

✅ Tokenizer loaded (vocab: 28,996)

📝 Sub-word demo: 'nephroblastoma' → ['ne', '##ph', '##ro', '##blast', '##oma']
   Only the FIRST sub-token gets the NER label; rest get -100


In [10]:
MAX_LENGTH = 128

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=MAX_LENGTH,
        is_split_into_words=True,  # Input is already word-tokenized
        padding=False,
    )
    all_labels = []
    for i in range(len(examples["tokens"])):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids, prev_word_idx = [], None
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)           # [CLS] / [SEP]
            elif word_idx != prev_word_idx:
                label_ids.append(examples["ner_tags"][i][word_idx])  # First sub-token
            else:
                label_ids.append(-100)           # Continuation sub-token
            prev_word_idx = word_idx
        all_labels.append(label_ids)
    tokenized["labels"] = all_labels
    return tokenized

print("⏳ Tokenizing datasets...")
tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels, batched=True,
    remove_columns=raw_datasets["train"].column_names
)
print(f"✅ Done — Train:{len(tokenized_datasets['train'])} Val:{len(tokenized_datasets['validation'])} Test:{len(tokenized_datasets['test'])}")

# Alignment verification
print("\n📋 Alignment check (first sample, first 15 tokens):")
ids = tokenized_datasets['train'][0]['input_ids']
lbls = tokenized_datasets['train'][0]['labels']
toks = tokenizer.convert_ids_to_tokens(ids)
print(f"{'Token':22s} | {'Label ID':10s} | Label Name")
print("-" * 48)
for tok, lbl in zip(toks[:15], lbls[:15]):
    name = LABEL_NAMES[lbl] if lbl >= 0 else "[IGNORED]"
    flag = " 🔵" if lbl > 0 else (" ✖️" if lbl == -100 else "")
    print(f"{tok:22s} | {str(lbl):10s} | {name}{flag}")

⏳ Tokenizing datasets...


Map:   0%|          | 0/5433 [00:00<?, ? examples/s]

Map:   0%|          | 0/924 [00:00<?, ? examples/s]

Map:   0%|          | 0/941 [00:00<?, ? examples/s]

✅ Done — Train:5433 Val:924 Test:941

📋 Alignment check (first sample, first 15 tokens):
Token                  | Label ID   | Label Name
------------------------------------------------
[CLS]                  | -100       | [IGNORED] ✖️
I                      | 0          | O
##dent                 | -100       | [IGNORED] ✖️
##ification            | -100       | [IGNORED] ✖️
of                     | 0          | O
AP                     | 0          | O
##C                    | -100       | [IGNORED] ✖️
##2                    | -100       | [IGNORED] ✖️
,                      | 0          | O
a                      | 0          | O
ho                     | 0          | O
##mo                   | -100       | [IGNORED] ✖️
##logue                | -100       | [IGNORED] ✖️
of                     | 0          | O
the                    | 0          | O


## 5. Model Initialization & Metrics

In [11]:
print(f"⏳ Loading BioBERT for token classification...")
fine_tune_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS,
    id2label=ID2LABEL, label2id=LABEL2ID,
    ignore_mismatched_sizes=True,  # Replace classification head
)
total = sum(p.numel() for p in fine_tune_model.parameters())
print(f"✅ BioBERT loaded — {total:,} total params | head: {NUM_LABELS} labels")

⏳ Loading BioBERT for token classification...


pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-v1.1
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ BioBERT loaded — 107,721,987 total params | head: 3 labels


In [12]:
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_preds = [
        [LABEL_NAMES[pred] for pred, lbl in zip(prediction, label) if lbl != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [LABEL_NAMES[lbl] for lbl in label if lbl != -100]
        for label in labels
    ]
    results = seqeval.compute(
        predictions=true_preds, references=true_labels,
        mode="strict", scheme="IOB2"  # Entity-level strict F1
    )
    return {
        "precision": round(results["overall_precision"], 4),
        "recall":    round(results["overall_recall"], 4),
        "f1":        round(results["overall_f1"], 4),
        "accuracy":  round(results["overall_accuracy"], 4),
    }

print("✅ seqeval strict F1 metric ready")
print("   (Token accuracy ~98% is misleading — most tokens are 'O'. Use F1.)")

✅ seqeval strict F1 metric ready
   (Token accuracy ~98% is misleading — most tokens are 'O'. Use F1.)


## 6. Trainer Configuration

In [13]:
OUTPUT_DIR = "./biobert-disease-ner"

# Calculate warmup_steps from warmup_ratio
num_train_epochs = 4
per_device_train_batch_size = 16
train_dataset_size = len(tokenized_datasets["train"])
num_training_steps = int(num_train_epochs * (train_dataset_size / per_device_train_batch_size))
warmup_steps = int(0.1 * num_training_steps) # 0.1 is the original warmup_ratio

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=warmup_steps, # Using warmup_steps instead of warmup_ratio
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    fp16=(device == 'cuda'),
    report_to="none",
    seed=42,
)

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer, padding=True, label_pad_token_id=-100
)

trainer = Trainer(
    model=fine_tune_model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
print("✅ Trainer ready | epochs=4 | lr=2e-5 | batch=16 | fp16=" + str(device == 'cuda'))

✅ Trainer ready | epochs=4 | lr=2e-5 | batch=16 | fp16=True


## 7. Evaluate BEFORE Training

⚠️ **Must run before `trainer.train()`** to capture the random-head baseline for a meaningful before/after delta.

In [14]:
print("📊 Evaluating BEFORE fine-tuning (random classification head)...")
print("   Expected: near-zero F1")
metrics_before = trainer.evaluate(eval_dataset=tokenized_datasets["test"])
print("\n📋 PRE-TRAINING (Test Set):")
for k in ["eval_precision","eval_recall","eval_f1","eval_accuracy"]:
    print(f"  {k:20s}: {metrics_before.get(k, 0):.4f}")

📊 Evaluating BEFORE fine-tuning (random classification head)...
   Expected: near-zero F1



📋 PRE-TRAINING (Test Set):
  eval_precision      : 0.0212
  eval_recall         : 0.0229
  eval_f1             : 0.0220
  eval_accuracy       : 0.5857


## 8. Fine-Tune BioBERT

In [15]:
print("🚀 Starting fine-tuning...")
t_start = time.time()
train_result = trainer.train()
t_total = time.time() - t_start

print(f"\n✅ Training complete in {t_total/60:.1f} min | loss: {train_result.training_loss:.4f}")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"   Model saved → {OUTPUT_DIR}/")

🚀 Starting fine-tuning...


Epoch,Training Loss,Validation Loss,Model Preparation Time,Precision,Recall,F1,Accuracy
1,0.044849,0.045901,0.060500,0.820100,0.839900,0.829900,0.984900
2,0.027575,0.045363,0.060500,0.826900,0.843700,0.835200,0.985600
3,0.014786,0.046567,0.060500,0.840400,0.883100,0.861200,0.986900
4,0.007548,0.054160,0.060500,0.844300,0.881800,0.862600,0.987400


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


✅ Training complete in 3.5 min | loss: 0.0551


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

   Model saved → ./biobert-disease-ner/


In [16]:
print("📊 Evaluating AFTER fine-tuning...")
metrics_after = trainer.evaluate(eval_dataset=tokenized_datasets["test"])
print("\n📋 POST-TRAINING (Test Set):")
for k in ["eval_precision","eval_recall","eval_f1","eval_accuracy"]:
    print(f"  {k:20s}: {metrics_after.get(k, 0):.4f}")

📊 Evaluating AFTER fine-tuning...



📋 POST-TRAINING (Test Set):
  eval_precision      : 0.8586
  eval_recall         : 0.9042
  eval_f1             : 0.8808
  eval_accuracy       : 0.9866


In [17]:
# Before / After delta table
keys = ["precision","recall","f1","accuracy"]
labels_disp = ["Precision","Recall","F1 (seqeval strict)","Token Accuracy"]
before_vals = [metrics_before.get(f"eval_{k}", 0) for k in keys]
after_vals  = [metrics_after.get(f"eval_{k}", 0)  for k in keys]
deltas = [a - b for a, b in zip(after_vals, before_vals)]

delta_df = pd.DataFrame({
    "Metric":          labels_disp,
    "Before Training": [f"{v:.4f}" for v in before_vals],
    "After Training":  [f"{v:.4f}" for v in after_vals],
    "Δ":               [f"+{d:.4f}" if d > 0 else f"{d:.4f}" for d in deltas],
    "Δ%":              [f"+{d*100:.1f}%" if d > 0 else f"{d*100:.1f}%" for d in deltas],
})

print("\n" + "="*65)
print("📈 BEFORE vs AFTER FINE-TUNING — NCBI Disease Test Set")
print("="*65)
print(delta_df.to_string(index=False))
f1_delta = deltas[2]
print(f"\n💡 Fine-tuning improved entity F1 by {f1_delta*100:.1f} percentage points.")
print("   Token accuracy is misleadingly high because most tokens are 'O'.")


📈 BEFORE vs AFTER FINE-TUNING — NCBI Disease Test Set
             Metric Before Training After Training       Δ     Δ%
          Precision          0.0212         0.8586 +0.8374 +83.7%
             Recall          0.0229         0.9042 +0.8813 +88.1%
F1 (seqeval strict)          0.0220         0.8808 +0.8588 +85.9%
     Token Accuracy          0.5857         0.9866 +0.4009 +40.1%

💡 Fine-tuning improved entity F1 by 85.9 percentage points.
   Token accuracy is misleadingly high because most tokens are 'O'.


## 9. Inference on KHCC Samples — Baseline vs Fine-Tuned

In [18]:
finetuned_pipe = pipeline(
    "ner", model=OUTPUT_DIR, tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=0 if device == 'cuda' else -1
)
print("✅ Fine-tuned pipeline loaded")

finetuned_results = run_ner_pipeline(PATHOLOGY_SAMPLES, finetuned_pipe, score_threshold=0.60)
print("✅ Fine-tuned inference complete")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Fine-tuned pipeline loaded
✅ Fine-tuned inference complete


In [19]:
print("="*75)
print("🔬 SIDE-BY-SIDE: BASELINE vs FINE-TUNED")
print("="*75)

comparison_rows = []
for sample, base, ft in zip(PATHOLOGY_SAMPLES, baseline_results, finetuned_results):
    print(f"\n[{sample['id']}] {sample['site']}")
    print(f"TEXT: {sample['text'][:100]}...")
    print(f"\n{'BASELINE':45s} | FINE-TUNED")
    print("─"*45 + "─┼─" + "─"*35)
    base_ents = [f"{e['word']} [{e['entity_group']}] ({e['score']:.2f})" for e in base["entities"]]
    ft_ents   = [f"{e['word']} [{e['entity_group']}] ({e['score']:.2f})" for e in ft["entities"]]
    for j in range(max(len(base_ents), len(ft_ents), 1)):
        b = base_ents[j] if j < len(base_ents) else ""
        f_ = ft_ents[j]  if j < len(ft_ents)  else ""
        print(f"{b:45s} | {f_}")

    base_r = base["matched"] / base["total_gold"]
    ft_r   = ft["matched"]   / ft["total_gold"]
    delta_str = f"+{(ft_r-base_r):.0%}" if ft_r >= base_r else f"{(ft_r-base_r):.0%}"
    print(f"\nGold: {sample['gold_entities']}")
    print(f"Recall: {base_r:.0%} → {ft_r:.0%}  [{delta_str}] | Latency: {base['latency_ms']} ms → {ft['latency_ms']} ms")
    comparison_rows.append({
        "Sample": sample["id"], "Site": sample["site"],
        "Baseline Entities": len(base["entities"]), "FT Entities": len(ft["entities"]),
        "Baseline Recall": f"{base_r:.0%}", "FT Recall": f"{ft_r:.0%}",
        "Δ Recall": delta_str,
    })

print("\n" + "="*75)
print(pd.DataFrame(comparison_rows).to_string(index=False))

🔬 SIDE-BY-SIDE: BASELINE vs FINE-TUNED

[KHCC-001] Breast
TEXT: Sections show invasive ductal carcinoma, grade III, with extensive lymphovascular invasion. There is...

BASELINE                                      | FINE-TUNED
──────────────────────────────────────────────┼────────────────────────────────────
invasive [Detailed_description] (1.00)        | invasive ductal carcinoma [Disease] (0.98)
ductal carcino [Disease_disorder] (0.79)      | ductal carcinoma in situ [Disease] (0.99)
l [Biological_structure] (0.94)               | metastatic carcinoma [Disease] (0.97)
ductal carcinoma [Disease_disorder] (1.00)    | fibrocystic changes [Disease] (0.87)
solid [Detailed_description] (1.00)           | a [Disease] (0.95)
cr [Biological_structure] (0.93)              | ##crine metaplasia [Disease] (0.90)
##ib [Detailed_description] (0.73)            | 
ax [Biological_structure] (1.00)              | 
##illa [Biological_structure] (0.83)          | 
##ym [Biological_structure] (1.00)    

## 10. Stretch Challenge — SQLite JSON Export

Export all predictions with entity details, confidence scores, and latency to SQLite for EHR integration, audit trails, or downstream analysis.

In [20]:
DB_PATH = "/content/khcc_ner_predictions.db"

conn = sqlite3.connect(DB_PATH)
conn.executescript("""
    CREATE TABLE IF NOT EXISTS prediction_runs (
        run_id     INTEGER PRIMARY KEY AUTOINCREMENT,
        model_name TEXT, model_type TEXT, run_time TEXT, eval_f1 REAL, notes TEXT
    );
    CREATE TABLE IF NOT EXISTS samples (
        sample_id TEXT PRIMARY KEY, site TEXT, text TEXT, gold_entities TEXT
    );
    CREATE TABLE IF NOT EXISTS predictions (
        pred_id      INTEGER PRIMARY KEY AUTOINCREMENT,
        run_id       INTEGER, sample_id TEXT,
        entity_word  TEXT, entity_group TEXT, confidence REAL,
        char_start   INTEGER, char_end INTEGER, latency_ms REAL, entity_json TEXT
    );
    CREATE TABLE IF NOT EXISTS evaluation_metrics (
        metric_id  INTEGER PRIMARY KEY AUTOINCREMENT,
        run_id     INTEGER, split TEXT,
        precision  REAL, recall REAL, f1 REAL, accuracy REAL, recorded_at TEXT
    );
""")
conn.commit()
print(f"✅ SQLite schema ready: {DB_PATH}")

✅ SQLite schema ready: /content/khcc_ner_predictions.db


In [21]:
now = datetime.now().isoformat()

# Insert samples
for s in PATHOLOGY_SAMPLES:
    conn.execute("INSERT OR REPLACE INTO samples VALUES (?,?,?,?)",
                 (s["id"], s["site"], s["text"], json.dumps(s["gold_entities"])))

# Baseline run
conn.execute("INSERT INTO prediction_runs (model_name,model_type,run_time,eval_f1,notes) VALUES (?,?,?,?,?)",
             ("d4data/biomedical-ner-all","baseline",now,None,"Off-the-shelf biomedical NER"))
baseline_run_id = conn.execute("SELECT last_insert_rowid()").fetchone()[0]

for r in baseline_results:
    for e in r["entities"]:
        # Convert numpy.float32 score to standard Python float for JSON serialization
        e["score"] = float(e["score"])
        conn.execute(
            "INSERT INTO predictions (run_id,sample_id,entity_word,entity_group,confidence,char_start,char_end,latency_ms,entity_json) VALUES (?,?,?,?,?,?,?,?,?)",
            (baseline_run_id, r["id"], e["word"], e["entity_group"], e["score"], e["start"], e["end"], r["latency_ms"], json.dumps(e))
        )

# Fine-tuned run
ft_f1 = metrics_after.get("eval_f1", 0)
conn.execute("INSERT INTO prediction_runs (model_name,model_type,run_time,eval_f1,notes) VALUES (?,?,?,?,?)",
             ("dmis-lab/biobert-v1.1","finetuned",now,ft_f1,"BioBERT fine-tuned on NCBI Disease"))
ft_run_id = conn.execute("SELECT last_insert_rowid()").fetchone()[0]

for r in finetuned_results:
    for e in r["entities"]:
        # Convert numpy.float32 score to standard Python float for JSON serialization
        e["score"] = float(e["score"])
        conn.execute(
            "INSERT INTO predictions (run_id,sample_id,entity_word,entity_group,confidence,char_start,char_end,latency_ms,entity_json) VALUES (?,?,?,?,?,?,?,?,?)",
            (ft_run_id, r["id"], e["word"], e["entity_group"], e["score"], e["start"], e["end"], r["latency_ms"], json.dumps(e))
        )

# Metrics
for run_id, metrics, tag in [(baseline_run_id, metrics_before, "test_before"), (ft_run_id, metrics_after, "test_after")]:
    conn.execute(
        "INSERT INTO evaluation_metrics (run_id,split,precision,recall,f1,accuracy,recorded_at) VALUES (?,?,?,?,?,?,?)",
        (run_id, tag, metrics.get("eval_precision"), metrics.get("eval_recall"),
         metrics.get("eval_f1"), metrics.get("eval_accuracy"), now)
    )

conn.commit()
print(f"✅ All predictions exported to SQLite")

✅ All predictions exported to SQLite


In [22]:
# Useful queries
print("\n[Q1] Top-10 highest-confidence fine-tuned predictions:")
print(pd.read_sql("SELECT sample_id,entity_word,entity_group,ROUND(confidence,3) AS conf FROM predictions WHERE run_id=? ORDER BY confidence DESC LIMIT 10", conn, params=(ft_run_id,)).to_string(index=False))

print("\n[Q2] Entity count — baseline vs fine-tuned per sample:")
print(pd.read_sql("""SELECT p.sample_id,
    SUM(CASE WHEN r.model_type='baseline'  THEN 1 ELSE 0 END) baseline,
    SUM(CASE WHEN r.model_type='finetuned' THEN 1 ELSE 0 END) finetuned
FROM predictions p JOIN prediction_runs r ON p.run_id=r.run_id
GROUP BY p.sample_id""", conn).to_string(index=False))

print("\n[Q3] Before/after evaluation metrics:")
print(pd.read_sql("""SELECT r.model_type, em.split, ROUND(em.precision,4) precision,
    ROUND(em.recall,4) recall, ROUND(em.f1,4) f1
FROM evaluation_metrics em JOIN prediction_runs r ON em.run_id=r.run_id""", conn).to_string(index=False))

print("\n[Q4] Average latency per model:")
print(pd.read_sql("""SELECT r.model_type, ROUND(AVG(p.latency_ms),1) avg_ms,
    ROUND(MIN(p.latency_ms),1) min_ms, ROUND(MAX(p.latency_ms),1) max_ms
FROM predictions p JOIN prediction_runs r ON p.run_id=r.run_id GROUP BY r.model_type""", conn).to_string(index=False))

conn.close()


[Q1] Top-10 highest-confidence fine-tuned predictions:
sample_id                         entity_word entity_group  conf
 KHCC-005                                   G      Disease 0.996
 KHCC-001            ductal carcinoma in situ      Disease 0.987
 KHCC-002 adenocarcinoma of the sigmoid colon      Disease 0.987
 KHCC-003                                  Lu      Disease 0.985
 KHCC-005            ##lioblastoma multiforme      Disease 0.984
 KHCC-004     Clear cell renal cell carcinoma      Disease 0.983
 KHCC-001           invasive ductal carcinoma      Disease 0.977
 KHCC-001                metastatic carcinoma      Disease 0.967
 KHCC-003                 ##ng adenocarcinoma      Disease 0.964
 KHCC-005                 geographic necrosis      Disease 0.962

[Q2] Entity count — baseline vs fine-tuned per sample:
sample_id  baseline  finetuned
 KHCC-001        15          6
 KHCC-002        21          2
 KHCC-003        18          4
 KHCC-004         9          2
 KHCC-005        1

In [23]:
# JSON export for portability
JSON_PATH = "/content/khcc_ner_predictions.json"
export = {
    "export_time": datetime.now().isoformat(),
    "evaluation": {
        "before": {k.replace("eval_",""):v for k,v in metrics_before.items() if "eval_" in k},
        "after":  {k.replace("eval_",""):v for k,v in metrics_after.items()  if "eval_" in k},
    },
    "samples": [
        {"id": s["id"], "site": s["site"], "text": s["text"],
         "gold_entities": s["gold_entities"],
         "baseline":  {"entities": b["entities"], "latency_ms": b["latency_ms"]},
         "finetuned": {"entities": f["entities"], "latency_ms": f["latency_ms"]}}
        for s,b,f in zip(PATHOLOGY_SAMPLES, baseline_results, finetuned_results)
    ]
}
with open(JSON_PATH,"w") as jf:
    json.dump(export, jf, indent=2)
print(f"✅ JSON export: {JSON_PATH} ({os.path.getsize(JSON_PATH)/1024:.1f} KB)")

✅ JSON export: /content/khcc_ner_predictions.json (22.2 KB)


## 11. Discussion: BioBERT vs OpenAI at KHCC

### Dimension 1 — Cost

**BioBERT (fine-tuned):** One-time training cost of ~$0.50–2.00 on Colab T4. Marginal inference cost is essentially zero on self-hosted GPU infrastructure. At 100K pathology reports/year, the cost is flat and dominated by hardware, not per-call fees.

**OpenAI GPT-4:** A 300-word pathology report costs ~$0.005/call with GPT-4o. At 100K reports/year that becomes $500/year — significant at scale and unpredictable as prompt complexity grows.

**Verdict:** BioBERT wins decisively at KHCC's production scale.

---

### Dimension 2 — Privacy

**BioBERT:** Runs 100% on-premises. PHI never leaves KHCC's network. Fully compliant with Jordan's health data regulations. No BAA or third-party processing agreement required.

**OpenAI GPT-4:** PHI must be sent to US servers. Requires de-identification before every API call plus a Business Associate Agreement. Azure OpenAI with a private endpoint mitigates this but doubles infrastructure cost.

**Verdict:** BioBERT is the only viable option for identifiable patient records without significant compliance overhead.

---

### Dimension 3 — Latency

**BioBERT:** 30–80 ms on T4 GPU; 200–600 ms on CPU. Deterministic, zero network overhead. Suitable for real-time EHR population at the point of care.

**OpenAI GPT-4:** 3,000–15,000 ms API response time, plus 50–200 ms network round-trip from Amman. Batch API cuts costs 50% but introduces up to 24-hour turnaround — unsuitable for real-time workflows.

**Verdict:** BioBERT is 50–100× faster. Non-negotiable for live EHR integration.

---

### Dimension 4 — Accuracy

**BioBERT (NCBI Disease):** ~85–88% entity F1 (seqeval strict) on standardized disease mentions. Highly precise span boundaries. Reproducible — zero variance between identical calls.

**OpenAI GPT-4:** 85–92% F1 with few-shot prompting. Better at rare/novel disease names not well-represented in NCBI training data. However, temperature > 0 introduces response variance and results cannot be benchmarked cleanly with seqeval.

**Verdict:** Near parity. BioBERT has the edge on known terminology; GPT-4 edges ahead on rare entities and contextual reasoning.

---

### Dimension 5 — Schema Flexibility

**BioBERT:** Fixed label set (O / B-Disease / I-Disease). Adding new entity types requires re-annotation and retraining — typically 2–5 days of work. Excellent for stable, well-defined production tasks.

**OpenAI GPT-4:** Schema changes require only a prompt edit. Can extract diseases, medications, TNM stage, laterality, and negation simultaneously in a single call with `response_format=json`. Zero retraining required.

**Verdict:** GPT-4 wins decisively. This is its strongest practical advantage for clinical NLP research at KHCC where requirements evolve weekly.

---

### Dimension 6 — Hallucination Risk

**BioBERT:** Extractive model — output tokens must exist verbatim in the input text. Cannot hallucinate entity words. Only failure mode is misclassification (wrong label on a real span), which is detectable and auditable.

**OpenAI GPT-4:** Generative model — observed failure modes include inventing plausible-sounding disease names not present in the document, conflating similar diagnoses in ambiguous contexts, and incorrect ICD-10 normalization. Temperature=0 and few-shot examples reduce but do not eliminate this risk.

**Verdict:** BioBERT is fundamentally safer for patient-safety-critical pipelines. A hallucinated disease name in a tumor registry creates incorrect diagnosis codes, billing errors, and potential treatment plan deviations.

---

### Summary Scorecard

| Dimension | BioBERT (Fine-tuned) | OpenAI GPT-4 | Winner |
|---|---|---|---|
| Cost (at scale) | ~$0 / report | ~$0.005 / report | **BioBERT** |
| Privacy (PHI) | On-premises | Needs de-ID + BAA | **BioBERT** |
| Latency | 30–80 ms | 3,000–15,000 ms | **BioBERT** |
| Accuracy (F1) | ~85–88% | ~85–92% | Tie / GPT-4 |
| Schema flexibility | Requires retraining | Prompt-only | **GPT-4** |
| Hallucination risk | Near zero | Low–moderate | **BioBERT** |

---

### KHCC Deployment Recommendation

**Use BioBERT (fine-tuned) for:**
1. **Production tumor registry coding** — automated ICD-O coding at scale with full PHI compliance
2. **Real-time EHR integration** — sub-100ms latency is a hard requirement; GPT-4 cannot meet it
3. **High-volume retrospective review** — 50K+ historical reports where API costs become prohibitive
4. **Safety-critical decision support** — zero hallucination tolerance for pipelines feeding treatment decisions

**Use OpenAI GPT-4 for:**
1. **Research prototyping** — new extraction schemas where requirements change weekly
2. **Complex relational extraction** — negation (`no lymph node involvement`), uncertainty, multi-entity relationships
3. **Rare entity types** — newly described syndromes not in NCBI Disease training data
4. **Combined extraction + summarization** — tumor board reports needing both structured fields and narrative

**Recommended KHCC Architecture — Hybrid Cascade:**

```
Pathology Report
      │
      ▼
[BioBERT NER]  ←── Fast, private, on-prem (handles ~90% of cases)
      │
      ├─ Confidence ≥ 0.70 ──► [Structured Output → EHR/Registry]
      │
      └─ Confidence < 0.70 ──► [GPT-4 Review] (de-identified text only)
                                      │
                                      ▼
                               [Structured Output → EHR/Registry]
```

This cascade combines BioBERT's cost, privacy, and latency advantages with GPT-4's superior handling of ambiguous edge cases — while minimizing PHI exposure to the minimum necessary.

## 12. Final Summary

In [24]:
print("="*65)
print("📋 FINAL SUMMARY — BioBERT Disease NER @ KHCC")
print("="*65)

before_f1 = metrics_before.get("eval_f1", 0)
after_f1  = metrics_after.get("eval_f1", 0)
print(f"\n📈 Entity F1: {before_f1:.4f} → {after_f1:.4f}  (+{(after_f1-before_f1)*100:.1f} pp)")

base_recall = np.mean([r['matched']/r['total_gold'] for r in baseline_results])
ft_recall   = np.mean([r['matched']/r['total_gold'] for r in finetuned_results])
print(f"🎯 KHCC partial recall: {base_recall:.0%} → {ft_recall:.0%}")

base_lat = np.mean([r['latency_ms'] for r in baseline_results])
ft_lat   = np.mean([r['latency_ms'] for r in finetuned_results])
print(f"⏱  Avg latency: baseline {base_lat:.0f} ms | fine-tuned {ft_lat:.0f} ms")
print(f"\n🏥 Recommendation: Fine-tuned BioBERT for production | GPT-4 for research")
print(f"📁 Model: ./biobert-disease-ner/")
print(f"📁 DB:    /content/khcc_ner_predictions.db")
print(f"📁 JSON:  /content/khcc_ner_predictions.json")
print("="*65)

📋 FINAL SUMMARY — BioBERT Disease NER @ KHCC

📈 Entity F1: 0.0220 → 0.8808  (+85.9 pp)
🎯 KHCC partial recall: 95% → 61%
⏱  Avg latency: baseline 122 ms | fine-tuned 49 ms

🏥 Recommendation: Fine-tuned BioBERT for production | GPT-4 for research
📁 Model: ./biobert-disease-ner/
📁 DB:    /content/khcc_ner_predictions.db
📁 JSON:  /content/khcc_ner_predictions.json
